In [0]:
import os, sys, importlib
from collections import Counter

# ---- Parameters (the scheduled job sets these; widgets let you run it by hand) ----
dbutils.widgets.dropdown("mode", "daily", ["daily", "backfill"], "Mode")
dbutils.widgets.text("datasets", "", "Datasets (comma list, empty = all)")
dbutils.widgets.text("zones", "", "Zones (comma list, empty = all)")
dbutils.widgets.text("max_calls", "", "Max API calls (empty = no limit)")
dbutils.widgets.dropdown("plan_only", "false", ["false", "true"], "Plan only (no API calls)")

mode      = dbutils.widgets.get("mode")
datasets  = [s.strip() for s in dbutils.widgets.get("datasets").split(",") if s.strip()] or None
zones     = [s.strip() for s in dbutils.widgets.get("zones").split(",") if s.strip()] or None
max_calls = int(dbutils.widgets.get("max_calls")) if dbutils.widgets.get("max_calls").strip() else None
plan_only = dbutils.widgets.get("plan_only") == "true"

# ---- Imports from the repo's src folder ----
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import entsoe_client, entsoe_parser, silver_loader, orchestrator
for m in (entsoe_client, entsoe_parser, silver_loader, orchestrator):
    importlib.reload(m)

# ---- Plan ----
calls = orchestrator.plan(spark, mode, datasets, zones)
by_dataset = Counter(c[0] for c in calls)
print(f"Mode: {mode} | planned calls: {len(calls)}")
for d, n in sorted(by_dataset.items()):
    print(f"  {d:<20} {n:>5}")
if calls:
    first, last = calls[0], calls[-1]
    print(f"First: {first[0]} {first[1]} {first[4]:%Y-%m-%d %H:%M} -> {first[5]:%Y-%m-%d %H:%M} UTC")
    print(f"Last:  {last[0]} {last[1]} {last[4]:%Y-%m-%d %H:%M} -> {last[5]:%Y-%m-%d %H:%M} UTC")

# ---- Run (fetch to bronze only; the silver task loads the files) ----
if plan_only:
    print("\nPlan only: no API calls made.")
else:
    try:
        token = dbutils.secrets.get("power-prices", "entsoe-token")
    except Exception:
        token = "dummy-token-for-dry-run"
        print("\nNo ENTSO-E token stored yet: running as a dry run (calls will be unauthorized).")
    result = orchestrator.run(spark, token, mode, datasets, zones, max_calls, load_silver=False)
    print()
    for k, v in result.items():
        print(f"{k:<16} {v}")

    # Fail the task (and stop downstream tasks) if the circuit breaker tripped
    if result["stopped"].startswith("circuit breaker"):
        raise RuntimeError(
            f"Ingestion stopped: {result['stopped']}. "
            "Check the ENTSO-E token secret and power_prices.ops.api_call_log."
        )

In [0]:
display(spark.sql("""
    SELECT year(local_date)                                          AS year,
           COUNT(*)                                                  AS hours,
           ROUND(AVG(price_eur_mwh), 2)                              AS avg_price,
           ROUND(MIN(price_eur_mwh), 2)                              AS min_price,
           ROUND(MAX(price_eur_mwh), 2)                              AS max_price,
           SUM(CASE WHEN price_eur_mwh < 0 THEN 1 ELSE 0 END)        AS negative_hours,
           CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(source_resolution))) AS resolutions
    FROM power_prices.silver.price
    WHERE zone_code = 'DE_LU'
    GROUP BY year(local_date)
    ORDER BY year
"""))

display(spark.sql("""
    SELECT period_start, outcome, http_status,
           ROUND(bytes / 1024 / 1024, 2) AS mb,
           ROUND(duration_ms / 1000, 1)  AS seconds
    FROM power_prices.ops.api_call_log
    WHERE dataset = 'day_ahead_price' AND series_key = 'DE_LU' AND outcome = 'ok'
    ORDER BY called_at DESC
    LIMIT 8
"""))